# **PREPROCESSING AND EDA**
## **Phần 0: Chuẩn bị thư viện và dữ liệu**

### **0.1. Chuẩn bị thư viện và thiết lập môi trường**
Để đảm bảo quy tắc "from scratch", ta gọi lệnh `torch.set_grad_enabled(False)` ngay từ đầu. Khi autograd bị vô hiệu hoá, mọi lời gọi vô tình tới `loss.backward()` sẽ lập tức báo lỗi. Cố định `SEED = 42` để đảm bảo tính tái lập của mọi thử nghiệm.

In [16]:
!pip install -U numpy pandas matplotlib seaborn scipy scikit-learn ucimlrepo
!pip install torch torchvision

In [17]:
import os, gzip, shutil, time, copy, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import pandas as pd

# Chuyen thu muc lam viec ve goc du an neu dang o plan/
if os.path.basename(os.getcwd()) == "plan":
    os.chdir("..")
print("Thu muc lam viec:", os.getcwd())

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Tắt autograd cho toàn bộ notebook: gradient hoàn toàn do ta tự tính
torch.set_grad_enabled(False)

print("PyTorch Version:", torch.__version__, "| Số lượng CPU threads:", torch.get_num_threads())

Thu muc lam viec: c:\Users\Lenovo\Downloads\Pattern-Recognition
PyTorch Version: 2.14.1+cpu | Số lượng CPU threads: 8


### **0.2. Tải dữ liệu MNIST**
Hàm `load_mnist` tải dữ liệu qua `torchvision.datasets.MNIST`. Nếu nguồn mặc định gặp sự cố kết nối, hàm sẽ tự động chuyển sang mirror lưu trữ trên GitHub để đảm bảo quá trình tải dữ liệu luôn thông suốt.

In [3]:
DATA_ROOT = "./data"
MNIST_FILES = ["train-images-idx3-ubyte", "train-labels-idx1-ubyte",
               "t10k-images-idx3-ubyte", "t10k-labels-idx1-ubyte"]
GITHUB_MIRROR = "https://raw.githubusercontent.com/fgnt/mnist/master/"


def _download_from_mirror(root):
    raw = os.path.join(root, "MNIST", "raw")
    os.makedirs(raw, exist_ok=True)
    for name in MNIST_FILES:
        dst = os.path.join(raw, name)
        if os.path.exists(dst):
            continue
        gz = dst + ".gz"
        urllib.request.urlretrieve(GITHUB_MIRROR + name + ".gz", gz)
        with gzip.open(gz, "rb") as f_in, open(dst, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)


def load_mnist(root=DATA_ROOT):
    try:
        train = torchvision.datasets.MNIST(root, train=True, download=True)
        test = torchvision.datasets.MNIST(root, train=False, download=True)
    except Exception as e:
        print("Nguồn mặc định của torchvision lỗi (", type(e).__name__, "), chuyển sang mirror GitHub")
        _download_from_mirror(root)
        train = torchvision.datasets.MNIST(root, train=True, download=False)
        test = torchvision.datasets.MNIST(root, train=False, download=False)
    return train.data, train.targets, test.data, test.targets


Xtr_raw, ytr_raw, Xte_raw, yte_raw = load_mnist()
print("Dữ liệu thô - Train:", tuple(Xtr_raw.shape), "| Test:", tuple(Xte_raw.shape))

100.0%
100.0%
100.0%
100.0%

Dữ liệu thô - Train: (60000, 28, 28) | Test: (10000, 28, 28)


### **0.3. Giới thiệu và khám phá tổng quan bộ dữ liệu MNIST ban đầu**

Bộ dữ liệu **MNIST (Modified National Institute of Standards and Technology)** là bộ dữ liệu kinh điển trong thị giác máy tính và nhận dạng mẫu:
* **Nguồn gốc:** Do Yann LeCun, Corinna Cortes và Christopher Burges tạo ra từ hai tập dữ liệu của Viện Tiêu chuẩn và Kỹ thuật Quốc gia Hoa Kỳ (NIST), gồm chữ số viết tay của học sinh trung học và nhân viên Cục Điều tra Dân số Hoa Kỳ.
* **Quy mô:** Gồm **70.000 ảnh** chữ số viết tay chia thành **60.000 ảnh huấn luyện** và **10.000 ảnh kiểm tra**.
* **Định dạng ảnh:** Mỗi mẫu là một ảnh xám (grayscale) đơn kênh kích thước **28 × 28 pixel**, đã được căn giữa (centered) theo trọng tâm điểm ảnh.
* **Cường độ điểm ảnh:** Mỗi pixel có giá trị là số nguyên không âm trong đoạn $[0, 255]$:
  * $0$: Nền đen (không có mực).
  * $255$: Nét chữ màu trắng (độ đậm mực tối đa).
* **Nhãn lớp:** Các số nguyên từ $0$ đến $9$, tương ứng với 10 chữ số trong hệ thập phân.

In [4]:
# Kiểm tra kiểu dữ liệu, kích thước và miền giá trị của bộ dữ liệu MNIST thô
print("Kiểu dữ liệu của Xtr_raw:      ", type(Xtr_raw))
print("Kiểu dữ liệu của ytr_raw:      ", type(ytr_raw))
print("Kích thước tập ảnh train (raw):", Xtr_raw.shape)
print("Kích thước tập nhãn train (raw):", ytr_raw.shape)
print("Kiểu phần tử (dtype) của ảnh:  ", Xtr_raw.dtype)
print("Kiểu phần tử (dtype) của nhãn: ", ytr_raw.dtype)
print(f"Miền giá trị pixel: [min = {Xtr_raw.min().item()}, max = {Xtr_raw.max().item()}]")
print("Danh sách các lớp nhãn (0-9):  ", torch.unique(ytr_raw).tolist())
print(f"Số lượng mẫu mỗi lớp (train):   {torch.bincount(ytr_raw).tolist()}")

Kiểu dữ liệu của Xtr_raw:       <class 'torch.Tensor'>
Kiểu dữ liệu của ytr_raw:       <class 'torch.Tensor'>
Kích thước tập ảnh train (raw): torch.Size([60000, 28, 28])
Kích thước tập nhãn train (raw): torch.Size([60000])
Kiểu phần tử (dtype) của ảnh:   torch.uint8
Kiểu phần tử (dtype) của nhãn:  torch.int64
Miền giá trị pixel: [min = 0, max = 255]
Danh sách các lớp nhãn (0-9):   [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Số lượng mẫu mỗi lớp (train):   [5923, 6742, 5958, 6131, 5842, 5421, 5918, 6265, 5851, 5949]


### **0.4. Phân chia dữ liệu và Chuẩn hoá (Tiền xử lý)**

**1. Phân chia tập dữ liệu:**
* 60.000 ảnh huấn luyện ban đầu được chia ngẫu nhiên thành **50.000 ảnh train** (dùng huấn luyện mô hình) và **10.000 ảnh validation** (dùng tinh chỉnh siêu tham số và early stopping).
* Tập **test** gồm 10.000 ảnh được giữ nguyên vẹn và **chỉ được đánh giá đúng một lần duy nhất** ở Phần 7 để đo lường năng lực tổng quát hoá thực tế của mô hình.

**2. Bản chất toán học của tiền xử lý (Z-score Standardization):**
Mỗi ảnh $28 \times 28$ được duỗi phẳng thành một vector đặc trưng $\mathbf{x} \in \mathbb{R}^{784}$ ($784 = 28 \times 28$).
Sau đó, các giá trị pixel được chia cho 255 để đưa về đoạn $[0, 1]$ và chuẩn hoá theo công thức Z-score:
$$x_{\text{norm}} = \frac{x / 255.0 - \mu_{\text{train}}}{\sigma_{\text{train}}}$$
Trong đó $\mu_{\text{train}}$ và $\sigma_{\text{train}}$ là giá trị trung bình và độ lệch chuẩn được tính **chỉ trên 50.000 ảnh của tập train**.
* **Nguyên tắc chống rò rỉ dữ liệu (No Data Leakage):** Tuyệt đối không dùng thông tin thống kê của tập validation hay test khi chuẩn hoá, nhằm phản ánh chính xác bài toán thực tế khi mô hình gặp các mẫu dữ liệu mới chưa từng biết.

In [5]:
g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(Xtr_raw), generator=g)
tr_idx, va_idx = perm[:50000], perm[50000:]

_X = Xtr_raw[tr_idx].reshape(-1, 784).float() / 255.0
MEAN, STD = _X.mean().item(), _X.std().item()


def preprocess(X_raw):
    # Duỗi phẳng ảnh thành vector 784 chiều, đưa về [0, 1] và chuẩn hoá Z-score theo MEAN, STD của train.
    return (X_raw.reshape(-1, 784).float() / 255.0 - MEAN) / STD


X_train, y_train = preprocess(Xtr_raw[tr_idx]), ytr_raw[tr_idx]
X_val, y_val = preprocess(Xtr_raw[va_idx]), ytr_raw[va_idx]
X_test, y_test = preprocess(Xte_raw), yte_raw

print(f"Giá trị thống kê tập train: MEAN = {MEAN:.4f}, STD = {STD:.4f}")
print("Kích thước sau tiền xử lý:")
print("  X_train:", tuple(X_train.shape), "| y_train:", tuple(y_train.shape))
print("  X_val:  ", tuple(X_val.shape), "  | y_val:  ", tuple(y_val.shape))
print("  X_test: ", tuple(X_test.shape), "  | y_test: ", tuple(y_test.shape))
print("Phân bố số mẫu mỗi lớp trên tập train (50.000 mẫu):", torch.bincount(y_train).tolist())

Giá trị thống kê tập train: MEAN = 0.1307, STD = 0.3082
Kích thước sau tiền xử lý:
  X_train: (50000, 784) | y_train: (50000,)
  X_val:   (10000, 784)   | y_val:   (10000,)
  X_test:  (10000, 784)   | y_test:  (10000,)
Phân bố số mẫu mỗi lớp trên tập train (50.000 mẫu): [4923, 5613, 4958, 5093, 4863, 4546, 4933, 5204, 4882, 4985]


### **0.5. Khám phá và kiểm định chất lượng dữ liệu dạng bảng (Tabular EDA)**
Chuyển đổi dữ liệu ảnh thành cấu trúc DataFrame bảng đặc trưng (784 pixel và 1 cột nhãn target) để kiểm định toàn diện chất lượng dữ liệu theo quy chuẩn Lab 02:
* Kích thước (`shape`), các dòng đầu tiên (`head()`), cấu trúc kiểu dữ liệu (`info()`), thống kê đặc trưng (`describe().T`).
* Tên lớp mục tiêu (`target_names`), giá trị khuyết thiếu (`missing values`), dòng trùng lặp (`duplicates`), đặc trưng hằng số (`zero variance`) và giá trị âm.

In [6]:
# Tạo DataFrame từ tập dữ liệu thô và kiểm tra kích thước
col_names = [f"pixel_{i}" for i in range(784)]
df_mnist = pd.DataFrame(Xtr_raw.reshape(-1, 784).numpy(), columns=col_names)
df_mnist['target'] = ytr_raw.numpy()

print("Số lượng mẫu:", df_mnist.shape[0])
print("Số lượng cột: ", df_mnist.shape[1])

Số lượng mẫu: 60000
Số lượng cột:  785


In [7]:
# 5 dòng đầu tiên của tập dữ liệu
print("\n5 dòng đầu tiên của tập dữ liệu:")
display(df_mnist.head())


5 dòng đầu tiên của tập dữ liệu:


,pixel_0,pixel_1,pixel_2,pixel_3,pixel_4,pixel_5,pixel_6,pixel_7,pixel_8,pixel_9,...,pixel_775,pixel_776,pixel_777,pixel_778,pixel_779,pixel_780,pixel_781,pixel_782,pixel_783,target
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,5
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,9


In [8]:
# Cấu trúc và kiểu dữ liệu của bộ dữ liệu
print("Thông tin của bộ dữ liệu:")
df_mnist.info()

Thông tin của bộ dữ liệu:
<class 'pandas.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Columns: 785 entries, pixel_0 to target
dtypes: int64(1), uint8(784)
memory usage: 45.3 MB


In [9]:
# Xem tên các lớp (target_name)
target_names = [f"{i}" for i in range(10)]
for idx, name in enumerate(target_names):
    print(f"Nhãn {idx} = '{name}'")

Nhãn 0 = '0'
Nhãn 1 = '1'
Nhãn 2 = '2'
Nhãn 3 = '3'
Nhãn 4 = '4'
Nhãn 5 = '5'
Nhãn 6 = '6'
Nhãn 7 = '7'
Nhãn 8 = '8'
Nhãn 9 = '9'


In [10]:
# Kiểm tra giá trị khuyết thiếu (missing values / NaN)
missing_counts = df_mnist.isna().sum()
print("Số lượng missing value từng cột")
print(missing_counts[missing_counts > 0] if missing_counts.sum() > 0 else "Không có missing value nào ở tất cả các cột.")
print("Tổng số giá trị thiếu trong toàn bộ dataset:", missing_counts.sum())

Số lượng missing value từng cột
Không có missing value nào ở tất cả các cột.
Tổng số giá trị thiếu trong toàn bộ dataset: 0


In [11]:
# Kiểm tra các dòng trùng lặp (Duplicate rows)
num_duplicates = df_mnist.duplicated().sum()
print(f"Số lượng dòng bị trùng lặp: {num_duplicates}")

Số lượng dòng bị trùng lặp: 0


In [12]:
# Kiểm tra đặc trưng hằng số (phương sai bằng 0) và giá trị âm
X_raw = df_mnist.drop(columns=['target'])

print(f"Số đặc trưng có phương sai = 0: {(X_raw.var() == 0).sum()}")
print(f"Số đặc trưng có giá trị âm:    {(X_raw < 0).any().sum()}")

Số đặc trưng có phương sai = 0: 67
Số đặc trưng có giá trị âm:    0


In [13]:
# Phân tích tên lớp và nhãn mục tiêu
for idx, name in enumerate(target_names):
    count = (df_mnist['target'] == idx).sum()
    print(f"Nhãn {idx} = '{name}': {count} mẫu ({count/len(df_mnist)*100:.2f}%)")

Nhãn 0 = '0': 5923 mẫu (9.87%)
Nhãn 1 = '1': 6742 mẫu (11.24%)
Nhãn 2 = '2': 5958 mẫu (9.93%)
Nhãn 3 = '3': 6131 mẫu (10.22%)
Nhãn 4 = '4': 5842 mẫu (9.74%)
Nhãn 5 = '5': 5421 mẫu (9.04%)
Nhãn 6 = '6': 5918 mẫu (9.86%)
Nhãn 7 = '7': 6265 mẫu (10.44%)
Nhãn 8 = '8': 5851 mẫu (9.75%)
Nhãn 9 = '9': 5949 mẫu (9.92%)


### **0.6. Lưu trữ dữ liệu đã xử lý phục vụ các Phase tiếp theo**
Dữ liệu sau khi xử lý được lưu vào thư mục `data/processed` dưới 2 định dạng rõ ràng, tiện lợi:
1. **File nén chuẩn NumPy (`mnist_processed.npz`):** Chứa sẵn `X_train`, `y_train`, `X_val`, `y_val`, `X_test`, `y_test` (chuẩn hoá Z-score) cùng `X_train_minmax`, `X_val_minmax`, `X_test_minmax` (dạng $[0, 1]$ cho Multinomial/Bernoulli NB) và thông số `MEAN, STD`. Các notebook Phase 1, Phase 2, Phase 3 có thể nạp ngay lập tức bằng `np.load()`.
2. **File bảng CSV (`mnist_val.csv`, `mnist_test.csv`, `mnist_train.csv`):** Dạng bảng văn bản mở được trực tiếp bằng Excel / VS Code / Pandas để người dùng dễ dàng xem và kiểm tra dữ liệu.

In [14]:
# Xoá sạch thư mục data/processed cũ trước khi tạo dữ liệu mới
PROCESSED_DIR = "./data/processed"
if os.path.exists(PROCESSED_DIR):
    shutil.rmtree(PROCESSED_DIR)
os.makedirs(PROCESSED_DIR, exist_ok=True)
print(f"Đã làm sạch và tạo mới thư mục: {PROCESSED_DIR}")

# 1. Lưu file npz tiện dụng cho Phase 1, Phase 2, Phase 3
X_train_minmax = _X.numpy()
X_val_minmax   = (Xtr_raw[va_idx].reshape(-1, 784).float() / 255.0).numpy()
X_test_minmax  = (Xte_raw.reshape(-1, 784).float() / 255.0).numpy()

npz_path = os.path.join(PROCESSED_DIR, "mnist_processed.npz")
np.savez_compressed(
    npz_path,
    X_train=X_train.numpy(),
    y_train=y_train.numpy(),
    X_val=X_val.numpy(),
    y_val=y_val.numpy(),
    X_test=X_test.numpy(),
    y_test=y_test.numpy(),
    X_train_minmax=X_train_minmax,
    X_val_minmax=X_val_minmax,
    X_test_minmax=X_test_minmax,
    MEAN=MEAN,
    STD=STD
)
print(f" Đã lưu file npz: {npz_path} ({os.path.getsize(npz_path)/(1024*1024):.2f} MB)")

# 2. Lưu file CSV dạng bảng người dùng có thể mở đọc trực tiếp
cols = [f"pixel_{i}" for i in range(784)]

def export_csv(X_arr, y_arr, filename):
    filepath = os.path.join(PROCESSED_DIR, filename)
    df_out = pd.DataFrame(np.round(X_arr, 4), columns=cols)
    df_out["target"] = y_arr
    df_out.to_csv(filepath, index=False)
    print(f" Đã lưu file CSV: {filepath} ({len(df_out)} dòng, {os.path.getsize(filepath)/(1024*1024):.2f} MB)")

export_csv(X_val.numpy(), y_val.numpy(), "mnist_val.csv")
export_csv(X_test.numpy(), y_test.numpy(), "mnist_test.csv")
export_csv(X_train.numpy(), y_train.numpy(), "mnist_train.csv")

Đã làm sạch và tạo mới thư mục: ./data/processed
 Đã lưu file npz: ./data/processed\mnist_processed.npz (34.82 MB)
 Đã lưu file CSV: ./data/processed\mnist_val.csv (10000 dòng, 58.46 MB)
 Đã lưu file CSV: ./data/processed\mnist_test.csv (10000 dòng, 58.46 MB)
 Đã lưu file CSV: ./data/processed\mnist_train.csv (50000 dòng, 292.26 MB)


In [15]:
# Kiểm tra nạp lại dữ liệu đã lưu để đảm bảo tính sẵn sàng cho Phase 1, 2, 3
loaded = np.load(npz_path)
print("Danh sách các mảng trong file npz:", list(loaded.keys()))
print("X_train shape:", loaded["X_train"].shape, "| y_train shape:", loaded["y_train"].shape)
print("X_val shape:  ", loaded["X_val"].shape,   "| y_val shape:  ", loaded["y_val"].shape)
print("X_test shape: ", loaded["X_test"].shape,  "| y_test shape: ", loaded["y_test"].shape)
print(f"Giá trị thống kê lưu trữ: MEAN = {loaded['MEAN']:.4f}, STD = {loaded['STD']:.4f}")

Danh sách các mảng trong file npz: ['X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test', 'X_train_minmax', 'X_val_minmax', 'X_test_minmax', 'MEAN', 'STD']
X_train shape: (50000, 784) | y_train shape: (50000,)
X_val shape:   (10000, 784) | y_val shape:   (10000,)
X_test shape:  (10000, 784) | y_test shape:  (10000,)
Giá trị thống kê lưu trữ: MEAN = 0.1307, STD = 0.3082
